# Phase 1: Colab Environment Setup

Set up the full SoARM + LIBERO + OpenVLA-OFT stack on a fresh Colab A100 runtime.

## Requirements

- [ ] ENV-01: All dependencies install in correct order without pip resolver conflicts
- [ ] ENV-02: EGL headless rendering configured — OffScreenRenderEnv produces non-black LIBERO frames
- [ ] ENV-03: OpenVLA-OFT loads on GPU (A100 bf16) and returns a valid 7-D action tensor

## Usage

**BLOCK A** (this block): Run cells 0-9 top to bottom, then restart the runtime.

**BLOCK B** (Plan 02): After restart, run verification cells for ENV-01 / ENV-02 / ENV-03.

> Note: Block A must complete fully before restarting. Do not run Block B cells before restart.

In [5]:
import os

# ── USER CONFIGURATION ──────────────────────────────────────────────────────
# Set REPO_ROOT to the path where you cloned SoARM-Research.
# If using Google Drive: "/content/drive/MyDrive/SoARM-Research"
# If using git clone directly to Colab: "/content/SoARM-Research"
REPO_ROOT = "/content/drive/MyDrive/SoARM-Research"
# ────────────────────────────────────────────────────────────────────────────

# Derived path constants (do not edit these)
LIBERO_ROOT = f"{REPO_ROOT}/LIBERO/libero/libero"
LIBERO_PKG  = f"{REPO_ROOT}/LIBERO"   # path to setup.py directory
OUT_DIR     = f"{REPO_ROOT}/LIBERO/notebooks/outputs"
BDDL_FILE   = (
    f"{LIBERO_ROOT}/bddl_files/libero_spatial/"
    "pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate.bddl"
)

# Create outputs directory so Block B render check can save there
os.makedirs(OUT_DIR, exist_ok=True)

print(f"REPO_ROOT   = {REPO_ROOT}")
print(f"LIBERO_ROOT = {LIBERO_ROOT}")
print(f"LIBERO_PKG  = {LIBERO_PKG}")
print(f"OUT_DIR     = {OUT_DIR}")
print(f"BDDL_FILE   = {BDDL_FILE}")
print(f"Saved → {OUT_DIR}  (outputs directory ready)")

REPO_ROOT   = /content/drive/MyDrive/SoARM-Research
LIBERO_ROOT = /content/drive/MyDrive/SoARM-Research/LIBERO/libero/libero
LIBERO_PKG  = /content/drive/MyDrive/SoARM-Research/LIBERO
OUT_DIR     = /content/drive/MyDrive/SoARM-Research/LIBERO/notebooks/outputs
BDDL_FILE   = /content/drive/MyDrive/SoARM-Research/LIBERO/libero/libero/bddl_files/libero_spatial/pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate.bddl
Saved → /content/drive/MyDrive/SoARM-Research/LIBERO/notebooks/outputs  (outputs directory ready)


In [6]:
# GPU assertion — D-06
# Check GPU availability and warn loudly if not A100.
# OpenVLA-OFT in bf16 requires ~16 GB VRAM; A100 (40 GB) is the target.
import torch

assert torch.cuda.is_available(), (
    "No GPU available. Go to Runtime > Change runtime type > Hardware accelerator > GPU."
)

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU:  {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

if "A100" not in gpu_name:
    print()
    print("WARNING: Expected A100, got", gpu_name)
    print("WARNING: OpenVLA-OFT in bf16 requires ~16 GB+ VRAM.")
    print("WARNING: ENV-03 will OOM on T4 (15 GB). Restart with an A100 runtime.")
    print("WARNING: You may still proceed for install-only testing on T4.")
else:
    print("A100 confirmed. Proceeding.")

GPU:  Tesla T4
VRAM: 15.6 GB



---

## BLOCK A: Install

Run all cells in this block **top to bottom**, then restart the runtime.

**Ordering is critical** — do not reorder or skip cells:

1. EGL system packages (apt) must install before pip mujoco
2. PyTorch must install before flash-attn (flash-attn compiles against torch CUDA headers)
3. The custom transformers fork must install last (prevents pip downgrade to PyPI version)

---

In [7]:
# Step 1 of 6 — EGL system packages
# Must run BEFORE pip mujoco install.
# These C libraries must exist when the mujoco Python extension builds.
# apt-get update refreshes repo index — prevents 404 for stale package URLs (e.g. libosmesa6).
!apt-get update -qq
!apt-get install -y -q --fix-missing \
    libglfw3 \
    libglew-dev \
    libosmesa6-dev \
    libgles2 \
    libglvnd0 \
    libegl-dev \
    libegl1 \
    libgl1-mesa-glx

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists...
Building dependency tree...
Reading state information...
libegl-dev is already the newest version (1.4.0-1).
libegl1 is already the newest version (1.4.0-1).
libgles2 is already the newest version (1.4.0-1).
libglvnd0 is already the newest version (1.4.0-1).
libglew-dev is already the newest version (2.2.0-4).
libglfw3 is already the newest version (3.3.6-1).
libosmesa6-dev is already the newest version (23.2.1-1ubuntu3.1~22.04.4).
libgl1-mesa-glx is already the newest version (23.0.4-0ubuntu1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 127 not upgraded.


In [8]:
# Step 2 of 6 — PyTorch 2.2.0 (cu121)
# Must run BEFORE flash-attn: flash-attn compiles CUDA kernels against installed torch headers.
!pip install torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 \
    --index-url https://download.pytorch.org/whl/cu121 -q

In [11]:
# Step 3 of 6 — MuJoCo + simulation stack
#
# numpy pin must be LAST in this cell (after all other installs settle).
# Version check uses subprocess, not import: torch already imported numpy 2.0.2
# in Cell 2 (GPU check), so sys.modules caches 2.0.2 in this session.
# After the runtime restart (before Block B) Python starts fresh and loads
# the on-disk numpy 1.26.4. The subprocess check reports the on-disk version.
import sys, subprocess
print(f"Python {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")

# mujoco 3.3.2: binary wheel for cp312; stable version for LIBERO
!pip install "mujoco==3.3.2" -q

# gym: --prefer-binary avoids source build on Python 3.12
!pip install "gym>=0.21,<=0.26" --prefer-binary -q

# robosuite MUST be 1.4.0 — 1.5.x removed SingleArmEnv which LIBERO requires
!pip install     robosuite==1.4.0     bddl==1.0.1     easydict==1.9     cloudpickle==2.1.0     einops==0.4.1     "imageio[ffmpeg]" -q

# opencv: <4.10 required — 4.10+ requires numpy>=2 (breaks torch 2.2.0)
!pip install "opencv-python-headless>=4.7,<4.10" --prefer-binary -q

# numpy LAST: torch 2.2.0 compiled against numpy 1.x headers.
# numpy 2.x breaks torch._ARRAY_API (torch.from_numpy fails — used in LIBERO).
# Resolver warnings about jax/cupy/opencv needing numpy>=2 are safe to ignore.
!pip install "numpy>=1.24,<2" --force-reinstall -q

import mujoco
mv = tuple(int(x) for x in mujoco.__version__.split("."))
print(f"mujoco {mujoco.__version__} installed")
if mv >= (3, 0, 0):
    print("✓ mujoco 3.x — bddl_base_domain.py mj_kinematics shim is active")

# Check the ON-DISK numpy version (not sys.modules which may cache old version).
# torch imported numpy 2.x in Cell 2; after restart Python loads the disk version.
np_ver = subprocess.check_output(
    [sys.executable, "-m", "pip", "show", "numpy"], text=True
)
for line in np_ver.splitlines():
    if line.startswith("Version:"):
        v = line.split()[1]
        print(f"numpy {v} on disk (fresh after restart)")
        assert v < "2", f"numpy {v} >= 2 on disk — re-run this cell"
        break

print("✓ simulation stack installed — restart runtime now (Runtime > Restart session)")


Python 3.12.13
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.6.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.1 which is incompatible.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4

AssertionError: numpy 2.0.2 >= 2 — torch.from_numpy will fail!

In [7]:
# Step 4 of 6 — LIBERO editable install
# LIBERO_PKG is defined in Cell 1. setup.py may be missing if Google Drive
# didn't sync the nested LIBERO git repo — auto-clone from GitHub in that case.
import os, subprocess, sys
from pathlib import Path

# Auto-clone LIBERO if setup.py not found on Drive
_libero_setup = Path(LIBERO_PKG) / 'setup.py'
if not _libero_setup.exists():
    _fallback = '/content/libero'
    print(f'⚠ {LIBERO_PKG}/setup.py missing (Google Drive may not sync nested git repos)')
    print(f'  Cloning LIBERO from GitHub → {_fallback} ...')
    _r = subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/Lifelong-Robot-Learning/LIBERO.git', _fallback],
        capture_output=True, text=True,
    )
    if _r.returncode != 0:
        raise RuntimeError(f'LIBERO clone failed:\n{_r.stderr}')
    # Redirect paths to cloned location (affects Block B cells)
    LIBERO_PKG  = _fallback
    LIBERO_ROOT = f'{_fallback}/libero/libero'
    BDDL_FILE   = (
        f'{LIBERO_ROOT}/bddl_files/libero_spatial/'
        'pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate.bddl'
    )
    print(f'✓ LIBERO cloned. Paths updated → LIBERO_PKG={LIBERO_PKG}')
else:
    print(f'✓ LIBERO found at {LIBERO_PKG}')

# editable install — LIBERO package changes are live without reinstall
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', LIBERO_PKG, '-q'],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])
else:
    print('✓ LIBERO installed in editable mode from:', LIBERO_PKG)

⚠ /content/drive/MyDrive/SoARM-Research/LIBERO/setup.py missing (Google Drive may not sync nested git repos)
  Cloning LIBERO from GitHub → /content/libero ...
✓ LIBERO cloned. Paths updated → LIBERO_PKG=/content/libero
✓ LIBERO installed in editable mode from: /content/libero


In [8]:
# Step 5 of 6 — OpenVLA-OFT supporting packages + custom transformers fork
# The git fork (moojink/transformers-openvla-oft) MUST be installed LAST in this cell.
# Do NOT install transformers from PyPI — the fork replaces it entirely.
# The fork adds bidirectional attention for parallel decoding; PyPI version lacks this.
!pip install \
    timm==0.9.10 \
    tokenizers==0.19.1 \
    sentencepiece==0.1.99 \
    peft==0.11.1 \
    accelerate \
    huggingface_hub -q

# Install transformers fork LAST — pip resolver cannot downgrade to PyPI version this way.
# Commit SHA comment for reproducibility: installs from main branch of the fork repo.
# To pin a specific commit: git+https://github.com/moojink/transformers-openvla-oft.git@<SHA>
!pip install git+https://github.com/moojink/transformers-openvla-oft.git -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 50.4 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.1 MB/s eta

In [4]:
# Step 6 of 6 — flash-attn (pre-built wheel — avoids 90-min T4 source compile)
#
# flash-attn 2.5.x has NO cp312 wheel. cp312 support was added in 2.6.x.
# flash-attn 2.6.3 has wheels for cu118 and cu123, but NOT cu121.
# cu118 wheels run on CUDA 12.x via CUDA forward compatibility.
#
# Wheel: flash_attn-2.6.3+cu118torch2.2cxx11abiFALSE-cp312-cp312-linux_x86_64.whl
# Install time: ~30 seconds vs 60-90 min source compile.
import subprocess, sys, torch

print(f"Detected: CUDA={torch.version.cuda}, Torch={torch.__version__}, Python=cp{sys.version_info.major}{sys.version_info.minor}")

!pip install packaging ninja -q

# cu118 + torch2.2 + cp312 — runs on CUDA 12.x (forward compat)
WHEEL = (
    "https://github.com/Dao-AILab/flash-attention/releases/download/"
    "v2.6.3/flash_attn-2.6.3+cu118torch2.2cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"
)
print(f"Installing: {WHEEL}")
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", WHEEL, "-q"],
    capture_output=True, text=True
)
if r.returncode != 0:
    print("Primary wheel failed, trying cu123 fallback...")
    WHEEL2 = (
        "https://github.com/Dao-AILab/flash-attention/releases/download/"
        "v2.6.3/flash_attn-2.6.3+cu123torch2.2cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"
    )
    r2 = subprocess.run(
        [sys.executable, "-m", "pip", "install", WHEEL2, "-q"],
        capture_output=True, text=True
    )
    if r2.returncode != 0:
        print(r2.stderr[-500:])
        raise RuntimeError("Both flash-attn wheels failed. Check https://github.com/Dao-AILab/flash-attention/releases/tag/v2.6.3")

import flash_attn
print(f"✓ flash-attn {flash_attn.__version__} installed")


Detected: CUDA=12.1, Torch=2.2.0+cu121, Python=cp312
Installing: https://github.com/Dao-AILab/flash-attention/releases/download/v2.6.3/flash_attn-2.6.3+cu118torch2.2cxx11abiFALSE-cp312-cp312-linux_x86_64.whl


ImportError: libcudart.so.11.0: cannot open shared object file: No such file or directory

In [3]:
import subprocess
subprocess.run(['git', 'pull'], cwd='/content/drive/MyDrive/SoARM-Research')

CompletedProcess(args=['git', 'pull'], returncode=128)

---

## *** STOP — Restart runtime now ***

Go to: **Runtime > Restart session** (or press Ctrl+M .), then continue from BLOCK B below.

Do **not** run any cells below this point until after the runtime has restarted.

After restart, continue in **this notebook** — run the BLOCK B cells below.

---